# Rover Process

**Yuxi Zhang** · Oct 18, 2026 · [@yuxi-research](https://github.com/yuxi-research)

📘 This notebook creates a DataFrame ready for neutron count correction. It matches neutron monitoring station data from the Neutron Monitor Database (NMDB; [https://www.nmdb.eu/nest/](https://www.nmdb.eu/nest/)) with rover timestamps for each survey. Each rover timestamp is matched to the nearest timestamp in the NM data, using a tolerance of 10 minutes. This allows the datasets to be matched even when the rover and NMDB timestamps are not exactly aligned.

⚙️ **Main Processing Steps**

1. **Process**

   * Match neutron monitoring station data with rover data.

💬 **Notes**

📥 **Input**

* `./data_lv1/{farm_name}/rover/`

  * `{farm_name}_{date}__rover_clean.shp`: Cleaned rover data.
* `settings.py`: Configuration file specifying the farm name, target CRS, folder structure, and file paths.

📤 **Output**

* `./data_lv2/{farm_name}/rover/`

  * `{farm_name}_{date}__rover_dataframe.shp`: Rover data matched with neutron monitoring flux data.

🧩 **Parameters**

* `pth_nmdb`: Update the NM station name or file path if needed. Make sure that the NM data have been downloaded manually. When using a different NM station, the reference flux and station coordinates used in `corrCounts.ipynb` must also be updated accordingly.


### Settings

In [1]:
%load_ext autoreload
%autoreload 2

import os
import pandas as pd
import geopandas as gpd
import datetime as dt
import matplotlib.pyplot as plt
import re

In [2]:
# Path settings
from global_settings import *
farm_name = os.environ.get("FARM_NAME", "Amaroo")
dirs = ProjectPaths(farm_name=farm_name)
proj_crs = dirs.proj_crs

### Read data

In [3]:
# Find all the rover files in the folder
filenames_rover = [f for f in os.listdir(dirs.lv1_rover) if re.match(rf'^{farm_name}_.*_rover_clean.csv$', f)]
filenames_rover.sort()
filenames_rover


In [4]:
# Read NMDB data
file_nmdb = dirs.raw_file_nmdb
df_nmdb = pd.read_csv(file_nmdb)
df_nmdb['DateTime (UTC)'] = pd.to_datetime(df_nmdb['DateTime (UTC)'], format='%Y-%m-%d %H:%M:%S')

In [5]:
# Read boundary
bound = gpd.read_file(os.path.join(dirs.lv1_shp, f'{farm_name}_boundary_epsg{proj_crs}.shp'))

In [6]:
# soil_all['Date'].unique()

### Loop rover files and merge rover data with neutron station data

In [7]:
log = []
for filename_rover in filenames_rover[0:]:
    farm_name, date_str = re.split( '[_|.]', filename_rover)[0:2]
    print(farm_name, date_str)

    # Read rover data
    df_rover = pd.read_csv(os.path.join(dirs.lv1_rover, filename_rover))
    df_rover['geometry'] = gpd.points_from_xy(df_rover['LongDec'], df_rover['LatDec'], crs= '4326').to_crs(proj_crs)
    df_rover['Date Time(UTC)'] = pd.to_datetime(df_rover['Date Time(UTC)'], format='%Y-%m-%d %H:%M:%S')

    # Merge rover data with NMDB data. 
    ## Not rounding timestamp. This will result in many missing values in NMDB data and is not used in the final version of the code. It is kept here for reference.
    # gdf_rover = df_rover.merge(df_nmdb, how='left', left_on='Date Time(UTC)', right_on='DateTime (UTC)'
    #                             ).drop(columns=['DateTime (UTC)'])

    # Round timestamp to the nearest according to the timestamps of NMDB data. Tolerance is set to 10 minutes. This helps when the rover data is not exactly aligned with the NMDB data in time.
    gdf_rover = pd.merge_asof(df_rover.sort_values('Date Time(UTC)'), df_nmdb.sort_values('DateTime (UTC)'),
                                left_on="Date Time(UTC)", right_on="DateTime (UTC)", 
                                direction="nearest", tolerance=pd.Timedelta("10min")
                                ).drop(columns=['DateTime (UTC)'])

    # Remove rows with missing NMDB data and convert to GeoDataFrame
    log.append(gdf_rover[gdf_rover['NMCounts'].isna()])
    gdf_rover = gdf_rover.dropna(subset='NMCounts')
    gdf_rover = gpd.GeoDataFrame(gdf_rover, geometry='geometry')

    # Check completness of neutron monitoring station data
    if len(df_rover) > len(gdf_rover):
        print(f"Warning: {len(df_rover)-len(gdf_rover)}/{len(df_rover)} neutron monitoring station data is not found. Please check the data file.")
    if len(gdf_rover) == 0:
        print("Error: No neutron station data was found.")
        continue


    gdf_rover.to_csv(dirs.lv2_data / f'{farm_name}_{date_str}_rover_dataframe.csv', index=False)
    print("File saved:", dirs.lv2_data / f'{farm_name}_{date_str}_rover_dataframe.csv')

